# Analysis of Degree Modifiers in Wiki talk-pages
## Outline:
- Start with some list of modifiers of degree
- open wiki talk pages and see if there are other hedges by asking gemini
- create final list of modifiers of degree by adding the found set
- extract all modifiers of degree based on string matching
- use gemini to filter out cases where modifiers of degree appear 
- add labels of the context in which the modifier is used including the following:
    - whether it's in a question
    - valence 
    - whether the modifier fuctions to soften the utterance, be formal, or be literal


Load libraries etc

In [105]:
import os
from google import genai
from dotenv import load_dotenv
import json
import pandas as pd
import xml.etree.ElementTree as ET
from google.genai import types
load_dotenv()
client = genai.Client(
    api_key=os.getenv("google_api_key"),
)
MAX_ENTRIES = 1000

### Decide on list of hedges

In [ ]:
# starting list of hedges
JP_DEGREE_MODIFIERS = [
    "ちょっと", "少し", "やや", "かなり", "非常に", "大変", "とても", "すごく", "めっちゃ", 
    "かなりの", "相当な", "極めて", "非常に多くの", "大いに", "大いなる", "大きな", 
    "大きく", "大きめの", "大きめに", "大きめな", "大きめで", "大きめだと"
]
EN_DEGREE_MODIFIERS = [
    "a bit", "a little", "somewhat", "fairly", "quite", "very", "extremely", "highly", "greatly", 
    "considerably", "substantially", "significantly", "markedly", "notably", "exceptionally", "remarkably", "tremendously", "immensely", "hugely", "enormously", "massively", "overwhelmingly", "profoundly", "exceedingly", "extraordinarily", "unusually", "abundantly", "plentifully", "copiously", "awfully", "terribly", "horribly", "dreadfully", "atrociously", "appallingly", "shockingly", "staggeringly", "stunningly", "breathtakingly", "mind-blowingly", "jaw-droppingly", "eye-poppingly", "heart-stoppingly", "spine-tinglingly", "hair-raisingly", "blood-curdlingly", "bone-chillingly"
]

In [10]:
# prompt for additional hedging extraction

prompt = """You are analyzing English text to find degree modifiers that are NOT already in a known list.

DEFINITION: A degree modifier is a word or short phrase that scales the intensity of a gradable adjective, adverb, or verb — it answers "to what extent / how much" (e.g., "very good," "barely visible," "completely wrong," "kind of slow"). Do NOT count:
- frequency/habituality adverbs (often, rarely, usually, typically, generally, mostly)
- epistemic/evidential adverbs marking certainty or source of a claim (probably, apparently, obviously, allegedly, supposedly, arguably, definitely, presumably, possibly)
- evaluative stance adverbs commenting on the whole proposition (surprisingly, worryingly, unfortunately, ideally)
- plain manner adverbs (quickly, carefully) unless they are clearly functioning to scale intensity rather than describe how an action was performed

TASK:
1. Read the text below.
2. Identify every word or short phrase functioning as a degree modifier.
3. Exclude any modifier that appears (in any form/case) in the EXCLUSION LIST below.
4. For each remaining (new) modifier found, report: the modifier itself, the word/phrase it modifies, and the exact quote/sentence it appears in.
5. If a modifier is ambiguous (could be degree or something else depending on context), include it but note the ambiguity.
6. If no new modifiers are found, return an empty array — do not force matches.

EXCLUSION LIST (already known — do not report these):
Overly, Excessively, Truly, Barely, Hardly, Scarcely, Just, Merely, Only, A tad, A touch, Marginally, Minimally, Nominally, Mildly, Vaguely, Kind of, Rather, Entirely, Somewhat, Considerably, Substantially, Significantly, Notably, Remarkably, Noticeably, Appreciably, Highly, Incredibly, Extremely, Unbelievably, Very, Exceptionally, Extraordinarily, Unusually, Immensely, Tremendously, Enormously, Hugely, Greatly, Vastly, Completely, Absolutely, Utterly, Totally, Wholly, Fully, Thoroughly, Perfectly, Supremely, Infinitely, Profoundly, Intensely, Strongly, Comparatively, Relatively, Proportionally, Mega, Ultra, So, Real, Way, Way too, Dead, Well, Mad, Hella, Wicked, Bloody, Damn, Freaking, Crazy, Crazily, Measurably, Quantifiably, Reasonably, Moderately, A little, A little bit, A bit, Slightly, Sort of, Kinda, Sorta, A whisker, A smidge, A smidgen, A hair, Faintly, Literally, Positively, Downright, Plain, Monumentally, Staggeringly, Astonishingly, Astoundingly, Phenomenally, Spectacularly, Stunningly, Breathtakingly, Achingly, Quite, Almost, Nearly, Practically, Virtually, Essentially, Basically, Just about, All but, Lil', Awfully, Terribly, Frightfully, Painfully, Deathly, Blindingly, Insanely, Ridiculously, Stupidly, Sick, Proper, Jolly, Fricking, Effing, Goddamn, Seriously, Stupid

TEXT TO ANALYZE:

{text}


OUTPUT FORMAT: Return strict JSON only — a single array containing ONE object per modifier found. If the text contains multiple new modifiers, include all of them as separate objects in the same array; do not merge them or return only one. Example shape with two entries:
[
  {"modifier": "...", "modifies": "...", "context": "...", "ambiguous": true/false},
  {"modifier": "...", "modifies": "...", "context": "...", "ambiguous": true/false}...
]
If exactly one modifier is found, return an array with just that one object. If none are found, return []. Do not include commentary outside the JSON."""

In [11]:
# check the first 1000 pages to see that we are not missing any modifiers of degree which occur relatively frequently.
namespace = {}
count = 0
# set directory to one above where this code is
# set to current directory where this file is
# create jsonl in one above current directory in /data
# set current directory to where this notebook is
with open('/Users/yuka/Documents/Academics/Stanford/Research/crossCulturalPoliteness/comparingCulturesPolitenessLLM/finalized_pipeline/data/en_check_hedge_comprehensiveness.jsonl', 'w') as f:
    os.chdir('/Users/yuka/Documents/Academics/Stanford/Research/crossCulturalPoliteness/comparingCulturesPolitenessLLM/wiki_corpus/enwiki_data')
    for file_string in ['enwiki-latest-pages-meta-current27.xml-p78975910p80475909', 'enwiki-latest-pages-meta-current27.xml-p81975910p83084748', 'enwiki-latest-pages-meta-current27.xml-p81975910p83327930']:
        for event, elem in ET.iterparse(file_string, events=['start','end']): #events=['end'] means levaves parsed before parent
            tag = elem.tag.split('}')[-1] if '}' in elem.tag else elem.tag
        # # create progress bar for count
        # pbar = tqdm(total=100)
            if tag == 'namespaces':
                for child in elem:
                    child_tag = child.tag.split('}')[-1] if '}' in child.tag else child.tag
                    namespace[child.attrib['key']] = child.text
            if event == 'end' and tag == 'page': # {website link}fins would not work. it has to be {namespace}ns
                if elem.find('{*}ns').text == '3': # this should be all user talk pages
                    # print all children of elem
                    for child in elem:
                        content = child.tag.split('}')[-1] if '}' in child.tag else child.tag
                        if content == 'revision':
                            timestamp = child.find('{*}timestamp').text if child.find('{*}timestamp') is not None else None
                            # print("timestamp:", timestamp)
                            comment = child.find('{*}comment').text if child.find('{*}comment') is not None else None
                            # print("comment:", comment)
                            text = child.find('{*}text').text if child.find('{*}text') is not None else None
                            # print("text:", text)
                            # break text up by newlines and parse each line for hedging language
                            if text is not None:
                                # split for newline
                                prompt_text = prompt.replace("{text}", text)
                                # Build a unique, traceable key so results can be mapped
                                # back to the source page/revision after the batch completes.
                                key = f"request-{count}"
 
                                request_obj = {
                                    "key": key,
                                    "request": {
                                        "contents": [
                                            {"parts": [{"text": prompt_text}], "role": "user"}
                                        ]
                                    }
                                }
                                f.write(json.dumps(request_obj, ensure_ascii=False) + "\n")
                                count += 1
                                break
                elem.clear()
    print(count)

877953


In [13]:
# only take lines of file which are 0 mod 10
with open('/Users/yuka/Documents/Academics/Stanford/Research/crossCulturalPoliteness/comparingCulturesPolitenessLLM/finalized_pipeline/data/en_check_hedge_comprehensiveness.jsonl', 'r') as f:
    lines = f.readlines()
    selected_lines = [line for i, line in enumerate(lines) if i % 10 == 0]
# save smaller file
with open('/Users/yuka/Documents/Academics/Stanford/Research/crossCulturalPoliteness/comparingCulturesPolitenessLLM/finalized_pipeline/data/en_check_hedge_comprehensiveness_small.jsonl', 'w') as f:
    f.writelines(selected_lines)    

In [14]:
uploaded_file = client.files.upload(
    file='/Users/yuka/Documents/Academics/Stanford/Research/crossCulturalPoliteness/comparingCulturesPolitenessLLM/finalized_pipeline/data/en_check_hedge_comprehensiveness_small.jsonl',
    config=types.UploadFileConfig(display_name='name', mime_type='jsonl')
)
input_file = uploaded_file.name

file_batch_job = client.batches.create(
    model="gemini-3-flash-preview",
    src=input_file,
    config={
        'display_name': "en_check_hedge_comprehensiveness",
    },
)

print(f"Created batch job: {file_batch_job.name}")


Created batch job: batches/ehzidbpzfpuef1gs9ytwogiqyomriilz17cc


In [ ]:
# check batch status
for batch in client.batches.list():
    print(f"Batch job {batch.display_name} status: {batch.state} file name: {batch.dest.file_name}")  
# read file
file_content = client.files.download(file="files/batch-ehzidbpzfpuef1gs9ytwogiqyomriilz17cc")

with open("result_en_check_hedge_comprehensiveness_small.jsonl", "wb") as f:
    f.write(file_content)

Batch job en_check_hedge_comprehensiveness status: JOB_STATE_SUCCEEDED file name: files/batch-ehzidbpzfpuef1gs9ytwogiqyomriilz17cc
Batch job jp-label-modifier-hedge-valence-fixed status: JOB_STATE_SUCCEEDED file name: files/batch-arhpd9xri9yyvw9u5tuyphcpypqooee38tc8
Batch job jp-label-modifier-hedge-valence-fixed status: JOB_STATE_SUCCEEDED file name: files/batch-45y92xkn89x6m1nqmeoal7mtcq8bnv221aec
Batch job en-label-modifier-hedge-valence-fixed status: JOB_STATE_SUCCEEDED file name: files/batch-2uewrg9yx76j63soy9f8vopkrfaietk0uy6h
Batch job en-label-modifier-hedge status: JOB_STATE_SUCCEEDED file name: files/batch-objaoavhw8v3qt2h0a5g9iczdk5hcneo6jsx
Batch job jp-label-modifier-hedge status: JOB_STATE_SUCCEEDED file name: files/batch-dq48s9ethhqxoc2yxdp71e0q3ena1eqrej0h
Batch job en-label-modifier-hedge status: JOB_STATE_SUCCEEDED file name: files/batch-v4l66qtwe0jejgn7y336ebppyx55qva5f3un


In [ ]:
# open first 10 lines of the result file
modifiers_to_add = list()
def extract_json_array(text):
    text = text.strip()
    if text.startswith("```"):
        text = text.strip("`")
        if text.lower().startswith("json"):
            text = text[4:]
        text = text.strip()
    return text
os.chdir('/Users/yuka/Documents/Academics/Stanford/Research/crossCulturalPoliteness/comparingCulturesPolitenessLLM/finalized_pipeline/data/results')
with open("result_en_check_hedge_comprehensiveness_small.jsonl", "r") as f:
    # read the whole file line by line
    for line in f.readlines():
        try:
            line = json.loads(line)

            if line['response']['candidates'][0]['content']['parts'][0]['text'] != "[]":
                line = json.loads(extract_json_array(line['response']['candidates'][0]['content']['parts'][0]['text']))
                for l in line:
                    print(l['modifier'],l['modifies'])
                modifiers_to_add.append((l['modifier'], l['modifies'], l['context'], l['ambiguous']))
        except Exception as e:
            print(f"Error processing line {i}: {e}")
            print(f"Line content: {line}")

more like an advertisement
strictly independent
more reliable
more independent
a few notches bring it down
the most enjoy
the most enjoy
directly based
fundamentally rewritten
unambiguously promotional
solely authored
far left
far left
that fast
fundamentally rewritten
strictly defined
strictly for discussing improvements
strictly for discussing improvements
adequately explaining
a small degree of generalisation
strictly for discussing improvements
strictly for discussing improvements
further working
poorly sourced
more encyclopedic
hard try
less than neutral
adequately explaining
broadly construed
more persistent
more strictly
strictly enforced
indefinitely blocked
strictly discussing improvements
a small degree of generalisation
strictly for discussing improvements
at all no citations included
confidently wrong
strictly for discussing improvements
strictly for discussing improvements to their associated main pages
most successful
most active
sufficiently notable
indefinitely blocked


TypeError: unsupported operand type(s) for +: 'function' and 'int'

In [64]:
# count and print modifiers from the most frequent to least frequent
from collections import Counter
modifier_counts = Counter([modifier for modifier, modifies, context, ambiguous in modifiers_to_add])
# count the number of times each modifier appears
for modifier, count in modifier_counts.most_common():
    print(f"{modifier}: {count}")

more: 4423
adequately: 2899
strictly: 2577
further: 2488
indefinitely: 1963
fundamentally: 1180
poorly: 821
closely: 772
the most: 691
in detail: 684
less than: 609
unduly: 604
most: 453
simply: 413
too: 324
much: 320
really: 314
genuinely: 252
exclusively: 158
sufficiently: 157
enough: 153
at all: 150
especially: 136
unsalvageably: 134
in full: 127
in any way: 126
directly: 106
right: 95
widely: 92
how: 90
in part: 87
Lightly: 77
less: 69
badly: 67
all: 67
a lot: 65
better: 64
pretty: 63
more than: 61
broadly: 53
at least: 49
exactly: 49
as: 46
solely: 44
far: 39
particularly: 37
heavily: 36
fairly: 35
purely: 34
zillions: 34
such: 33
unambiguously: 30
deeply: 28
that: 28
sincerely: 20
largely: 19
partially: 17
100%: 17
even: 17
as ... as possible: 16
Too: 16
outright: 13
suitably: 13
semi: 12
super: 12
as much as: 12
hard: 12
in some depth: 12
in depth: 12
little: 11
so much: 11
blatantly: 11
Most: 11
too much: 10
best: 10
shortly: 9
easily: 9
increasingly: 9
roughly: 9
effectively: 

In [ ]:
# final list of modifiers
EN_DEGREE_MODIFIERS = {'really', 'more', 'too', '', 'way', 'way too', 'dead', 'well', 'mad', 'hella', 'wicked', 'bloody', 'damn', 'freaking', 'crazy', 'crazily', 'measurably', 'quantifiably', 'reasonably', 'moderately', 'a little bit', 'a bit', 'slightly', 'sort of', 'kinda', 'sorta', 'a whisker', 'a smidge', 'a smidgen', 'a hair'}

## Extract strings from talk pages

## Gemini judges whether the strings extracted as 'hedges' are indeed used as degree modifiers

In [106]:
modifier_check_prompt_explanation = {
    '少し': """Explanation: The word 少し can function both as a modifier of degree ("a little"), but also have other functions. These are some examples and non-examples of its use as a modifier of degree:
- Example: 少し待ってください（＝しばらく）
- Example: 少し値上がりした（＝わずかに）
- Non-example: 少しも気にしない（＝「少しも＋否定」で「全く～ない」）""",

    '少しも': """Explanation: The word 少しも can function both as a modifier of degree, but also have other functions. These are some examples and non-examples of its use as a modifier of degree:
- Example: 少しも気にしない（＝「少しも＋否定」で「全く～ない」）
- Non-example: 少し気にしている（＝わずかに、通常の「少し」の用法）""",

    'ちょっと': """Explanation: The word ちょっと can function both as a modifier of degree ("a little/somewhat"), but also have other functions. These are some examples and non-examples of its use as a modifier of degree:
- Example: ちょっと高いですね（＝いくらか、控えめな程度を表す）
- Example: ちょっと待って（＝少し、控えめな程度を表す）
- Example: ちょっと分からない（＝少し、少々という意味）
- Non-example: 今日はちょっと…（＝婉曲的な断りを表す語用論的用法で、程度を表していない）
- Non-example: ちょっとしたプレゼントです（＝「ちょっとした＋名詞」で連体修飾）""",

    'それなり': """Explanation: The word それなり can function both as a modifier of degree ("in its own fitting way"), but also have other functions. These are some examples and non-examples of its use as a modifier of degree:
- Example: それなりに頑張った（＝自分の範囲・力なりに、副詞的用法）
- Non-example: それなりの理由がある（＝「それなりの＋名詞」で連体修飾）
- Non-example: 分量はそれなりになる（＝「なる」の補語としての名詞的用法）""",

    'すごく': """Explanation: The word すごく can function both as a modifier of degree ("very"), but also have other functions. These are some examples and non-examples of its use as a modifier of degree:
- Example: すごく美味しい（＝とても、口語的）
- Non-example: すごく！（＝一語での感嘆表現。間投詞的用法で何も修飾していない）""",

    'ずいぶん': """Explanation: The word 随分 can function both as a modifier of degree ("quite a lot"), but also have other functions. These are some examples and non-examples of its use as a modifier of degree:
- Example: 随分と変わったね（＝かなり、驚きを伴う変化）
- Non-example: 随分な言い方だね（＝ひどい・失礼な、「随分な＋名詞」でネガティブな評価的用法）""",

    'かなり': """Explanation: The word かなり can function both as a modifier of degree ("fairly/considerably"), but also have other functions. These are some examples and non-examples of its use as a modifier of degree:
- Example: かなり難しい（＝相当、程度副詞）
- Non-example: かなりの人数が集まった（＝「かなりの＋名詞」で連体修飾的用法）""",

    'いささか': """Explanation: The word いささか can function both as a modifier of degree ("somewhat/slightly"), but also have other functions. These are some examples and non-examples of its use as a modifier of degree:
- Example: いささか疑問が残る（＝少し、控えめな不満・懸念）
- Non-example: いささかも動じない（＝「いささかも＋否定」の呼応表現で全否定を強調）""",

    'ある程度': """Explanation: The word ある程度 can function both as a modifier of degree ("to some extent"), but also have other functions. These are some examples and non-examples of its use as a modifier of degree:
- Example: ある程度は理解できる（＝一定の範囲で）
- Non-example: ある程度の覚悟が必要だ（＝「ある程度の＋名詞」で連体修飾）""",

    'あまり': """Explanation: The word あまり can function both as a modifier of degree ("not very / to a certain degree"), but also have other functions. These are some examples and non-examples of its use as a modifier of degree:
- Example: あまり好きではない（＝「あまり＋否定」で「それほど～ない」という程度の低さを表す）
- Non-example: あまりにも突然で驚いたので声も出なかった（＝あまりに(も)＋形容詞で「過度に」を表す接続的用法）
- Non-example: 心配のあまり眠れなかった（＝「名詞＋のあまり」で名詞化した用法）""",

    'やたら': """Explanation: The word やたら can function both as a modifier of degree ("excessively"), but also have other functions. These are some examples and non-examples of its use as a modifier of degree:
- Example: やたら忙しい（＝過度に、否定的なニュアンスを伴う程度副詞）
- Non-example: やたらと人に話しかける（＝様態副詞、無差別に・むやみにという行動の仕方を表す）
- Non-example: むやみやたらに人に話しかける（＝「むやみやたらに＋動詞」で様態副詞的用法）""",

    'ちょっぴり': """Explanation: The word ちょっぴり can function both as a modifier of degree ("just a tiny bit"), but also have other functions. These are some examples and non-examples of its use as a modifier of degree:
- Example: ちょっぴり寂しい（＝ほんの少し、控えめな程度を表す）
- Non-example: ちょっぴりの勇気があれば大丈夫（＝「ちょっぴりの＋名詞」で連体修飾）""",

    '多少': """Explanation: The word 多少 can function both as a modifier of degree ("a little/somewhat"), but also have other functions. These are some examples and non-examples of its use as a modifier of degree:
- Example: 多少の誤差はある（＝いくらかの、名詞的用法「多少の＋名詞」）
- Example: 多少疲れた（＝少し）
- Example: 多少、表現などが似ていたかもしれない（＝少し）
- Example: 給料が多少上がった（＝わずかに、変化の度合い）
- Non-example: 多少にかかわらず（＝多いか少ないかに関係なく）""",

    '大変': """Explanation: The word 大変 can function both as a modifier of degree ("very"), but also have other functions. These are some examples and non-examples of its use as a modifier of degree:
- Example: 大変お世話になりました（＝非常に、丁寧な強調）
- Non-example: 本当に大変でした（＝苦労した、困難だった）
- Non-example: 他の方に大変な負担をかけている（＝「大変な＋名詞」で連体修飾）""",

    '相当': """Explanation: The word 相当 can function both as a modifier of degree ("considerably"), but also have other functions. These are some examples and non-examples of its use as a modifier of degree:
- Non-example: 10万円ほどに相当する（＝値する、動詞「相当する」）
- Example: 相当苦労した（＝かなり、副詞）
- Non-example: 相当な実力の持ち主だ（＝「相当な＋名詞」で連体修飾）""",

    '結構': """Explanation: The word 結構 can function both as a modifier of degree ("quite/fairly"), but also have other functions. These are some examples and non-examples of its use as a modifier of degree:
- Example: 結構難しい（＝かなり、程度副詞）
- Non-example: 結構な量の資料（＝かなりの、連体修飾）
- Non-example: お茶はもう結構です（＝十分なので不要、丁寧な断り）
- Non-example: とても結構なお品ですね（＝素晴らしい、褒め言葉）
- Non-example: 大雨の中でもけっこうする（＝「けっこう」だが漢字が違い「予定通り実行する」の意。紛らわしいので注意）""",

    '若干': """Explanation: The word 若干 can function both as a modifier of degree ("slightly/a few"), but also have other functions. These are some examples and non-examples of its use as a modifier of degree:
- Example: 若干不安が残る（＝少し）
- Example: 若干修正する必要がある（＝少し、わずかに）
- Non-example: 若干名を募集する（＝少々な人数）
- Non-example: 若干の修正が必要だ（＝「若干の＋名詞」で連体修飾）""",

    '非常に': """Explanation: The word 非常に can function both as a modifier of degree ("extremely"), but also have other functions. These are some examples and non-examples of its use as a modifier of degree:
- Example: 非常に重要だ（＝とても、フォーマルな場面）
- Example: 非常に危険な状況（＝極めて）
- Example: 非常に困った（＝すごく、困り具合の程度を表す）
- Non-example: 非常に備える（＝緊急事態）""",
    '十分': """Explanation: The word 十分 can function both as a modifier of degree ("enough/sufficiently"), but also have other functions. These are some examples and non-examples of its use as a modifier of degree:
- Example: 十分に注意する（＝十分に、程度副詞）
- Non-example: 十分な時間がある（＝足りる、名詞的用法「十分な＋名詞」）""",

    'わりと': """Explanation: The word 割と can function both as a modifier of degree ("relatively/fairly"), but also have other functions. These are some examples and non-examples of its use as a modifier of degree:
- Example: 割と簡単だった（＝思ったより、予想との比較）
- Example: 割と気に入っている（＝そこそこ、控えめな肯定）
- Non-example: 満席率は９割と、非常に混んでいる（＝割合が9割であるという数値を示す）""",

    'やや': """Explanation: The word やや can function both as a modifier of degree ("slightly"), but also have other functions. These are some examples and non-examples of its use as a modifier of degree:
- Example: やや値上がりした（＝少し、微妙な変化）
- Example: やや大きめのサイズ（＝若干）
- Non-example: ややこしい話だ（＝複雑だ。副詞「やや」とは語源が異なる形容詞）""",

    'とても': """Explanation: The word とても can function both as a modifier of degree ("very"), but also have other functions. These are some examples and non-examples of its use as a modifier of degree:
- Example: とても嬉しい（＝非常に、素直な肯定）
- Non-example: とても信じられない（＝「とても＋否定」で「到底～ない」という慣用的な否定強調構文）""",

    'とっても': """Explanation: The word とっても can function both as a modifier of degree ("very, casual"), but also have other functions. These are some examples and non-examples of its use as a modifier of degree:
- Example: とっても可愛い（＝「とても」のくだけた強調表現）
- Non-example: 私にとっても大事だよ。（＝「に＋とって＋も」で「〜に対しても」を表す別語。副詞の「とても」とは無関係の同形異義）
- Non-example: とっても！（＝一語での感嘆表現。何かを修飾しているわけではなく間投詞的用法）""",
}

prompt_instruction = """

Task: In this sentence, judge whether {modifier} is functioning as a modifier of degree.

{sentence_to_input}

If it is functioning as a modifier of degree, output the following in JSON format:
{"is_modifier": true, "modifies": "<the word or phrase it modifies>"}
otherwise, output:
{"is_modifier": false, "reason": "<a brief explanation of why it is not a modifier of degree>"}

Output only the raw JSON object above -- no markdown code fences (no ``` or ` characters), no extra commentary, and nothing before or after it. Use double quotes for every key and string value, and lowercase true/false for booleans."""

jp_string_extracted = pd.read_csv('/Users/yuka/Documents/Academics/Stanford/Research/crossCulturalPoliteness/comparingCulturesPolitenessLLM/finalized_pipeline/data/results/jp_hedged_sentencesJune26.csv')

import time

# 1. 1行ごとにインラインリクエストを作る（DataFrameの順序のまま）
inline_requests = []
for index, row in jp_string_extracted.iterrows():
    hedge = row['hedge']
    sentence = row['sentence']
    explanation = modifier_check_prompt_explanation.get(hedge, "")
    prompt_text = explanation + prompt_instruction.replace('{modifier}', hedge).replace('{sentence_to_input}', sentence)
    inline_requests.append({
        'contents': [{'parts': [{'text': prompt_text}], 'role': 'user'}]
    })
# save inline_requests to jsonl file
with open('/Users/yuka/Documents/Academics/Stanford/Research/crossCulturalPoliteness/comparingCulturesPolitenessLLM/finalized_pipeline/data/results/jp_hedge_modifier_check_inline_requests.jsonl', 'w') as f:
    for request in inline_requests:
        f.write(json.dumps(request, ensure_ascii=False) + "\n")
# 2. 1回のAPI呼び出しでバッチジョブを作成（ループでの逐次呼び出しの代わり）
batch_job = client.batches.create(
    model="gemini-3-flash-preview",
    src=inline_requests,
    config={'display_name': "hedge-modifier-check"},
)
print(f"Created batch job: {batch_job.name}")


Created batch job: batches/4y9kvhmrblr9blfqmhidxlf0ayuw7hl3vblf


In [ ]:
# 3. ジョブが終わるまでポーリング
completed_states = {"JOB_STATE_SUCCEEDED", "JOB_STATE_FAILED", "JOB_STATE_CANCELLED", "JOB_STATE_EXPIRED"}
while batch_job.state.name not in completed_states:
    print(f"Current state: {batch_job.state.name} -- waiting 30s")
    time.sleep(30)
    batch_job = client.batches.get(name=batch_job.name)

print(f"Job finished with state: {batch_job.state.name}")

# # 4. 提出した順序のままレスポンスを取得
# response_list = []
# for inline_response in batch_job.dest.inlined_responses:
#     if inline_response.response:
#         response_list.append(inline_response.response.text)
#     else:
#         print(f"Error: {inline_response.error}")
#         response_list.append(None)


{'is_modifier': False, 'reason': 'ここでの「やや」は、独立した副詞ではなく、「ややこしい（複雑な）」という形容詞の一部として使われています。'}


### save to file

In [ ]:
# create new columns in jp_string_extracted for is_modifier and modifies
jp_string_extracted['is_modifier'] = None
jp_string_extracted['modifies'] = None
for index, row in jp_string_extracted.iterrows():
    # get index'th element of response_list
    response = response_list[index]
    # print the response
    print(response[1:-1])
    print(f"Sentence: {row['sentence']}, Hedge: {row['hedge']}")
    # add is_modifier and modifies columns to jp_string_extracted
    try:
        response_json = json.loads(response[1:-1])
        jp_string_extracted.at[index, 'is_modifier'] = response_json.get('is_modifier', None)
        jp_string_extracted.at[index, 'modifies'] = response_json.get('modifies', None)
    except Exception as e:
        print(f"Error processing response for index {index}: {e}")
        jp_string_extracted.at[index, 'is_modifier'] = None
        jp_string_extracted.at[index, 'modifies'] = None
    if index==len(response_list)-1:
        break
jp_string_extracted.to_csv('/Users/yuka/Documents/Academics/Stanford/Research/crossCulturalPoliteness/comparingCulturesPolitenessLLM/finalized_pipeline/data/results/jp_hedged_sentencesJuly8_modifier_checked.csv', index=False)

{"is_modifier": false, "reason": "ここでの「やや」は、独立した副詞ではなく、「ややこしい（複雑な）」という形容詞の一部として使われています。"}
Sentence: 僕が（投稿者本人以外が）移動をすると、要約欄への書き込みなどいろいろややこしい部分があるので、できればあなたに移動をしていただきたいと思います。, Hedge: やや
{"is_modifier": true, "modifies": "忙しい"}
Sentence: また、少し忙しい身なので、早い回答を求めるならば、[[利用者:富沢順|利用者ページ]]に書いた代表と審議してください。, Hedge: 少し
{"is_modifier": true, "modifies": "仕事関係のことは考えたくない"}
Sentence: 専門は土木工学ですが、あまり仕事関係のことは考えたくないので、身近なことから書かせていただこうと思います。, Hedge: あまり
{"is_modifier": true, "modifies": "お世話になっている"}
Sentence: このウィキペディアには分からないことを読んで知ったり、知っている項目を執筆したりと大変お世話になっている者です。, Hedge: 大変
{"is_modifier": true, "modifies": "書き留めたい"}
Sentence: ちょっと何か書き留めたいというときにも是非ご活用ください。, Hedge: ちょっと
{"is_modifier": false, "reason": "In this sentence, "十分" is a component of the adjectival noun (na-adjective) "不十分" (insufficient), which functions as the predicate. It is not being used as an adverbial modifier to indicate the degree of another adjective or verb."}
Sentence: :*『この記事は検証可能な参考文献や出典が全く示されていないか、不十分です。, Hedge: 十分
Error processing 

## Add judgements of dimensions of context
### Valence

In [42]:
# add column to these csv files
# jp_df = pd.read_csv('.csv')
# en_df = pd.read_csv('en_hedged_sentences_extracted.csv')
import re
def ask(prompt: str) -> str:
    """Send a single prompt to Gemini and return the raw text response."""
    response = client.models.generate_content(model=MODEL, contents=prompt)
    return (response.text or "").strip()
 
 
def extract_number(text: str, default=None):
    """Pull the first integer/float out of a model response."""
    match = re.search(r"-?\d+(?:\.\d+)?", text)
    if match:
        return float(match.group())
    return default
def extract_field(text: str, field: str, default=None):
    """Pull a quoted-or-bare value for `field` out of a dict-like response."""
    match = re.search(
        rf"['\"]?{re.escape(field)}['\"]?\s*:\s*['\"]?([^,'\"}}]+)", text
    )
    if match:
        return match.group(1).strip()
    return default


In [ ]:
for row in response_list:

    # change response_list to extract is_modifier where row looks lie
    response_json = json.loads(row)
    is_modifier = response_json.get('response', False)
    row['is_modifier'] = is_modifier
    print(f"Sentence: {row['sentence']}, Hedge: {row['hedge']}, is_modifier: {is_modifier}")

{{'is_modifier': false, 'reason': 'ここでの「やや」は、独立した副詞ではなく、「ややこしい（複雑な）」という形容詞の一部として使われています。'}}


JSONDecodeError: Expecting property name enclosed in double quotes: line 1 column 2 (char 1)

### Check the following for each:
- (0) check if it is used as a modifier
- (1) if it is a question 
- (2) need for softening 
- (3) formality 
- (4) bearing on degree 
- (5) valence


In [52]:
def check_is_modifier(hedge: str, sentence: str) -> dict:
    """
    Judge whether `hedge` is functioning as a modifier of degree in `sentence`.

    Returns a dict: {'is_modifier': bool, 'detail': str, 'raw': str}
    where `detail` is the model's 'modifies' value when is_modifier is True,
    or its 'reason' value when False.
    """
    explanation = MODIFIER_CHECK_EXPLANATIONS.get(hedge, "")
    prompt = explanation + MODIFIER_CHECK_PROMPT_TEMPLATE.format(
        modifier=hedge, sentence=sentence
    )
    raw = ask(prompt)

    is_modifier_str = extract_field(raw, "is_modifier", default="false")
    is_modifier = is_modifier_str.lower().startswith("true")

    detail = extract_field(raw, "modifies") if is_modifier else extract_field(raw, "reason")

    return {"is_modifier": is_modifier, "detail": detail, "raw": raw}

def check_is_question(sentence: str) -> str:
    prompt = (
        "Is the following sentence a question? Please answer with 'Yes' or "
        f"'No'. Do not provide any additional explanation. sentence: {sentence}"
    )
    response = ask(prompt)
    return "Yes" if "Yes" in response else "No"
 
 
def rate_softener(hedge: str, sentence: str):
    prompt = (
        f"How much does the modifier '{hedge}' soften or harshen the nuance "
        "of the following sentence? Answer with a single number from 0 to 10, "
        "where 0 = makes the nuance much more harsh, 5 = no change, and "
        "10 = softens the nuance significantly. Respond with only the number, "
        f"no explanation. sentence: {sentence}"
    )
    return extract_number(ask(prompt))
 
 
def rate_formality(hedge: str, sentence: str):
    prompt = (
        f"How much does the modifier '{hedge}' change the formality/politeness "
        "of the following sentence? Answer with a single number from 0 to 10, "
        "where 0 = much less formal/polite, 5 = no change, and 10 = much more "
        "formal/polite. Respond with only the number, no explanation. "
        f"sentence: {sentence}"
    )
    return extract_number(ask(prompt))
 
 
def rate_degree(hedge: str, sentence: str):
    prompt = (
        f"How much literal information regarding degree or intensity does the "
        f"modifier '{hedge}' add to the following sentence? Answer with a "
        "single number from 0 to 5, where 0 = adds no literal degree "
        "information and 5 = adds substantial literal degree information. "
        f"Respond with only the number, no explanation. sentence: {sentence}"
    )
    return extract_number(ask(prompt))
 
 
def rate_valence(hedge: str, sentence: str):
    prompt = (
        f"In the following sentence, is the thing that '{hedge}' modifies "
        "framed as good/positive or bad/negative? Answer with a single number "
        "from 0 to 10, where 0 = very negative, 5 = neutral, and 10 = very "
        f"positive. Respond with only the number, no explanation. sentence: {sentence}"
    )
    return extract_number(ask(prompt))

In [ ]:
def process_dataframe(df: pd.DataFrame, max_entries: int = MAX_ENTRIES) -> pd.DataFrame:
    results = []
    for _, row in df.iterrows():
        hedge = row["hedge"]
        sentence = row["sentence"]
 
        if not check_is_modifier(hedge, sentence):
            continue
 
        result = {
            "hedge": hedge,
            "sentence": sentence,
            "is_question": check_is_question(sentence),
            "softener": rate_softener(hedge, sentence),
            "formality": rate_formality(hedge, sentence),
            "degree": rate_degree(hedge, sentence),
            "valence": rate_valence(hedge, sentence),
        }
        results.append(result)
        print(result)
 
        if len(results) >= max_entries:
            break
 
    return pd.DataFrame(results)
processed_jp_df = process_dataframe(jp_string_extracted)

NameError: name 'MODIFIER_CHECK_EXPLANATIONS' is not defined

## Generate frequency plots for each dimension

In [45]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DEFAULT_DIMENSIONS = ["softener", "formality", "degree", "valence"]


def plot_average_by_modifier(
    results_df: pd.DataFrame,
    dimensions=None,
    title: str = "Average dimension ratings by modifier",
    figsize=(12, 6),
):
    """
    results_df: the dataframe produced by process_dataframe (must contain a
        'hedge' column plus numeric columns for each dimension).
    dimensions: which numeric columns to plot as grouped bars. Defaults to
        softener/formality/degree/valence ('is_question' is categorical
        Yes/No, so it's excluded by default).
    """
    if dimensions is None:
        dimensions = [d for d in DEFAULT_DIMENSIONS if d in results_df.columns]

    # Average each dimension per hedge, preserving a stable modifier order.
    means = results_df.groupby("hedge", sort=False)[dimensions].mean()

    modifiers = means.index.tolist()
    n_modifiers = len(modifiers)
    n_dims = len(dimensions)

    x = np.arange(n_modifiers)
    bar_width = 0.8 / n_dims  # keep the group of bars within one x-slot

    fig, ax = plt.subplots(figsize=figsize)
    colors = plt.cm.tab10(np.linspace(0, 1, n_dims))

    for i, dim in enumerate(dimensions):
        offset = (i - (n_dims - 1) / 2) * bar_width
        ax.bar(x + offset, means[dim].values, width=bar_width, label=dim, color=colors[i])

    ax.set_xticks(x)
    ax.set_xticklabels(modifiers, rotation=45, ha="right")
    ax.set_xlabel("Modifier")
    ax.set_ylabel("Average rating")
    ax.set_title(title)
    ax.legend(title="Dimension")
    fig.tight_layout()

    return fig, ax